In [13]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [14]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git


Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 72 (delta 31), reused 43 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 344.94 KiB | 4.86 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [15]:
%cd /content/multilingual-rag-research


/content/multilingual-rag-research


In [16]:
!git config --global credential.helper store

In [17]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")

GitHub authentication configured.


In [18]:
!git fetch origin
!git switch nehna

Already on 'nehna'
Your branch is up to date with 'origin/nehna'.


In [19]:
import json
import random

with open("data/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("results/hybrid_top10.json", encoding="utf-8") as f:
    hybrid_results = json.load(f)

with open("results/predictions_hybrid.json", encoding="utf-8") as f:
    predictions = json.load(f)

print("Questions:", len(questions))
print("Corpus:", len(corpus))
print("Predictions:", len(predictions))

Questions: 100
Corpus: 1000
Predictions: 100


In [20]:
random.seed(7)
sample_questions = random.sample(questions, 30)

print("Manual sample size:", len(sample_questions))

Manual sample size: 30


In [21]:
def show_case(q):
    qid = q["question_id"]
    top_ids = hybrid_results[qid][:5]
    passages = [corpus[pid]["text"] for pid in top_ids]

    print("=" * 80)
    print("QUESTION:", q["question"])
    print("GOLD ANSWER(S):", q["gold_answers"])
    print("MODEL ANSWER:", predictions[qid])
    print("\nRETRIEVED PASSAGES:")
    for i, p in enumerate(passages, start=1):
        print(f"[{i}] {p[:300]}")
    print("=" * 80)

# try it on one
show_case(sample_questions[0])

QUESTION: What city is the capital of Victoria?
GOLD ANSWER(S): ['Melbourne', 'Melbourne', 'Melbourne']
MODEL ANSWER: Melbourne

RETRIEVED PASSAGES:
[1] Victoria (abbreviated as Vic) is a state in the south-east of Australia. Victoria is Australia's most densely populated state and its second-most populous state overall. Most of its population is concentrated in the area surrounding Port Phillip Bay, which includes the metropolitan area of its capit
[2] Prior to European settlement, the area now constituting Victoria was inhabited by a large number of Aboriginal peoples, collectively known as the Koori. With Great Britain having claimed the entire Australian continent east of the 135th meridian east in 1788, Victoria was included in the wider colon
[3] The economy of Victoria is highly diversified: service sectors including financial and property services, health, education, wholesale, retail, hospitality and manufacturing constitute the majority of employment. Victoria's total gross s

In [22]:
labels = []

In [83]:
labels.append({
    "question_id": sample_questions[29]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [84]:
show_case(sample_questions[30])

IndexError: list index out of range

In [89]:
from collections import Counter


In [90]:
faithfulness_counts = Counter(
    label["faithfulness"]
    for label in labels
)

print(faithfulness_counts)

Counter({'supported': 17, 'not_supported': 12, 'partial': 1})


In [91]:
total = len(labels)

for label, count in faithfulness_counts.items():

    percentage = (count / total) * 100

    print(
        f"{label}: {count}/{total} "
        f"({percentage:.1f}%)"
    )

not_supported: 12/30 (40.0%)
supported: 17/30 (56.7%)
partial: 1/30 (3.3%)


In [92]:
error_counts = Counter(
    label["error_type"]
    for label in labels
    if label["error_type"] is not None
)

print(error_counts)

Counter({'retrieval_missed': 10, 'generator_ignored_context': 2, 'none': 1})


In [93]:
total_errors = sum(error_counts.values())

print("Total labeled errors:", total_errors)

if total_errors > 0:

    for error_type, count in error_counts.items():

        percentage = (count / total_errors) * 100

        print(
            f"{error_type}: {count} "
            f"({percentage:.1f}%)"
        )

Total labeled errors: 13
retrieval_missed: 10 (76.9%)
generator_ignored_context: 2 (15.4%)
none: 1 (7.7%)


In [94]:
import re
import string

def normalize_answer(text):
    text = text.lower()

    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text
    )

    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )

    text = " ".join(text.split())

    return text

In [95]:
def exact_match(prediction, gold_answers):

    pred_norm = normalize_answer(prediction)

    for gold in gold_answers:

        if pred_norm == normalize_answer(gold):
            return 1

    return 0

In [96]:
for label in labels:

    qid = label["question_id"]

    q = next(
        q for q in questions
        if q["question_id"] == qid
    )

    pred = predictions[qid]

    em = exact_match(
        pred,
        q["gold_answers"]
    )

    if label["faithfulness"] == "supported" and em == 0:

        print("=" * 80)
        print("QUESTION:", q["question"])
        print("GOLD:", q["gold_answers"])
        print("MODEL:", pred)
        print("FAITHFULNESS:", label["faithfulness"])
        print("EM:", em)

QUESTION: What BBC radio station will carry the game in the United Kingdom?
GOLD: ['BBC Radio 5', 'Radio 5 Live', 'BBC Radio 5 Live']
MODEL: BBC Radio 5 Live and 5 Live Sports Extra
FAITHFULNESS: supported
EM: 0
QUESTION: When John Fox left as head coach for the Broncos, who replaced him?
GOLD: ['Gary Kubiak', 'Gary Kubiak', 'Kubiak']
MODEL: Gary Kubiak replaced John Fox as the new head coach.
FAITHFULNESS: supported
EM: 0
QUESTION: What did Baran develop during research at RAND 
GOLD: ['the concept of distributed adaptive message block switching', 'the concept of distributed adaptive message block switching', 'concept of distributed adaptive message block switching']
MODEL: distributed adaptive message block switching
FAITHFULNESS: supported
EM: 0
QUESTION: What might cause a higher student interest in learning the presented subject?
GOLD: ['Teacher enthusiasm', 'excitement', 'Teacher enthusiasm']
MODEL: Teacher enthusiasm contributes to a classroom atmosphere full of energy and enthu

In [97]:
for label in labels:

    qid = label["question_id"]

    q = next(
        q for q in questions
        if q["question_id"] == qid
    )

    pred = predictions[qid]

    em = exact_match(
        pred,
        q["gold_answers"]
    )

    if label["faithfulness"] == "not_supported" and em == 1:

        print("=" * 80)
        print("QUESTION:", q["question"])
        print("GOLD:", q["gold_answers"])
        print("MODEL:", pred)
        print("FAITHFULNESS:", label["faithfulness"])
        print("EM:", em)

QUESTION: What city is the capital of Victoria?
GOLD: ['Melbourne', 'Melbourne', 'Melbourne']
MODEL: Melbourne
FAITHFULNESS: not_supported
EM: 1
QUESTION: What did Tesla's design not need? 
GOLD: ['a commutator', 'commutator', 'a commutator']
MODEL: commutator
FAITHFULNESS: not_supported
EM: 1
QUESTION: When did the Venetians take total control of Cyprus?
GOLD: ['1489', '1489', '1489']
MODEL: 1489
FAITHFULNESS: not_supported
EM: 1
QUESTION: What percentage of oxygen is usually supplied by a medical mask?
GOLD: ['30%–50%', '30%–50%', '30%–50%', '30%–50%']
MODEL: 30%–50%
FAITHFULNESS: not_supported
EM: 1
QUESTION: Other than warships, what ships typically required high speeds?
GOLD: ['ocean liners', 'ocean liners', 'ocean liners']
MODEL: Ocean liners
FAITHFULNESS: not_supported
EM: 1
QUESTION: What was the estimated population of Pons Aelius around the 2nd century?
GOLD: ['2,000', '2,000', '2,000']
MODEL: 2,000
FAITHFULNESS: not_supported
EM: 1
QUESTION: By what name were the "supposedly

In [98]:
def f1_score(prediction, gold):

    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = {}

    for token in pred_tokens:
        common[token] = common.get(token, 0) + 1

    gold_counts = {}

    for token in gold_tokens:
        gold_counts[token] = gold_counts.get(token, 0) + 1

    overlap = 0

    for token, count in gold_counts.items():
        overlap += min(
            count,
            common.get(token, 0)
        )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )


def best_f1(prediction, gold_answers):

    return max(
        f1_score(prediction, gold)
        for gold in gold_answers
    )

In [99]:
for q in questions:

    qid = q["question_id"]

    pred = predictions[qid]

    em = exact_match(
        pred,
        q["gold_answers"]
    )

    f1 = best_f1(
        pred,
        q["gold_answers"]
    )

    if em == 0 and f1 > 0:

        print("=" * 80)
        print("QUESTION:", q["question"])
        print("GOLD:", q["gold_answers"])
        print("MODEL:", pred)
        print("EM:", em)
        print("F1:", round(f1, 3))

QUESTION: What did Pope Leo X launch against Luther?
GOLD: ['papal theologians and envoys', 'papal theologians and envoys', 'papal theologians and envoys']
MODEL: a series of papal theologians and envoys
EM: 0
F1: 0.8
QUESTION: What might cause a higher student interest in learning the presented subject?
GOLD: ['Teacher enthusiasm', 'excitement', 'Teacher enthusiasm']
MODEL: Teacher enthusiasm contributes to a classroom atmosphere full of energy and enthusiasm, leading to increased student interest and excitement in learning the subject matter.
EM: 0
F1: 0.167
QUESTION: What is the problem attributed to defining if two finite graphs are isomorphic?
GOLD: ['The graph isomorphism problem', 'graph isomorphism', 'The graph isomorphism problem']
MODEL: The computational problem of determining whether two finite graphs are isomorphic.
EM: 0
F1: 0.154
QUESTION: What did Davies want to build 
GOLD: ['proposed to build a nationwide network in the UK', 'a nationwide network', 'nationwide network

In [100]:
with open(
    "results/faithfulness_labels.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        labels,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved faithfulness labels!")

Saved faithfulness labels!


In [101]:
import os

print(
    os.path.exists(
        "results/faithfulness_labels.json"
    )
)

True


In [103]:
import os

print(
    "Notebook exists:",
    os.path.exists("notebooks/06_faithfulness.ipynb")
)

Notebook exists: False
